# 16 — Multithreading

## Objectives
- Create threads via `Thread` and `Runnable`
- Use `ExecutorService` and thread pools
- Understand synchronization and race conditions
- Apply `BlockingQueue` for producer-consumer

## Thread States
```
NEW → RUNNABLE → RUNNING → (BLOCKED/WAITING/TIMED_WAITING) → TERMINATED
```

In [1]:
import java.util.concurrent.*;
import java.util.concurrent.atomic.*;

// Atomic counter — thread-safe without synchronized
AtomicInteger counter = new AtomicInteger(0);

// Create 5 threads each incrementing 1000 times
ExecutorService pool = Executors.newFixedThreadPool(5);
List<Future<?>> futures = new ArrayList<>();

for (int i = 0; i < 5; i++) {
    final int threadId = i;
    futures.add(pool.submit(() -> {
        for (int j = 0; j < 1000; j++) counter.incrementAndGet();
        System.out.println("Thread-" + threadId + " done. Counter: " + counter.get());
    }));
}

// Wait for completion
for (Future<?> f : futures) { try { f.get(); } catch (Exception e) { e.printStackTrace(); } }
pool.shutdown();

System.out.println("Final counter (expected 5000): " + counter.get());

Thread-0 done. Counter: 1601
Thread-4 done. Counter: 5000
Thread-3 done. Counter: 4723
Thread-2 done. Counter: 4823
Thread-1 done. Counter: 4719
Final counter (expected 5000): 5000


## Mini Challenge
Implement a thread-safe `BoundedBuffer` that blocks producers when full and blocks consumers when empty.

In [2]:
import java.util.LinkedList;
import java.util.Queue;

public class BoundedBuffer<T> {
    private final Queue<T> buffer;
    private final int capacity;

    public BoundedBuffer(int capacity) {
        this.capacity = capacity;
        this.buffer = new LinkedList<>();
    }

    // Blocks if the buffer is full
    public synchronized void put(T item) throws InterruptedException {
        // Use a while loop to protect against spurious wakeups
        while (buffer.size() == capacity) {
            System.out.println(Thread.currentThread().getName() + " buffer full. Waiting...");
            wait(); 
        }
        
        buffer.add(item);
        System.out.println(Thread.currentThread().getName() + " produced: " + item);
        
        // Notify waiting consumers that an item is available
        notifyAll(); 
    }

    // Blocks if the buffer is empty
    public synchronized T take() throws InterruptedException {
        // Use a while loop to protect against spurious wakeups
        while (buffer.isEmpty()) {
            System.out.println(Thread.currentThread().getName() + " buffer empty. Waiting...");
            wait(); 
        }

        T item = buffer.poll();
        System.out.println(Thread.currentThread().getName() + " consumed: " + item);
        
        // Notify waiting producers that space has cleared up
        notifyAll(); 
        return item;
    }

    public synchronized int size() {
        return buffer.size();
    }
}

In [3]:
BoundedBuffer<Integer> q = new BoundedBuffer<>(2); // Capacity of 2

Thread producer = new Thread(() -> {
    try {
        for (int i = 1; i <= 5; i++) {
            q.put(i);
            Thread.sleep(100); // Simulate some production delay
        }
    } catch (InterruptedException e) {
        Thread.currentThread().interrupt();
    }
}, "Producer");

Thread consumer = new Thread(() -> {
    try {
        for (int i = 1; i <= 5; i++) {
            Thread.sleep(400); // Consumer is slower, forcing the buffer to fill up
            q.take();
        }
    } catch (InterruptedException e) {
        Thread.currentThread().interrupt();
    }
}, "Consumer");

consumer.start();
producer.start();

// Wait for both to finish in the notebook cell execution
producer.join();
consumer.join();
System.out.println("Challenge complete!");

Producer produced: 1
Producer produced: 2
Producer buffer full. Waiting...
Consumer consumed: 1
Producer produced: 3
Producer buffer full. Waiting...
Consumer consumed: 2
Producer produced: 4
Producer buffer full. Waiting...
Consumer consumed: 3
Producer produced: 5
Consumer consumed: 4
Consumer consumed: 5
Challenge complete!
